In [0]:
%pip install --force-reinstall databricks-automl-runtime==0.2.20.13
%pip install mlflow==2.20.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 443.8 kB/s eta 0:00:00
  Attempting uninstall: databricks-automl-runtime
    Found existing installation: databricks-automl-runtime 0.2.20
    Not uninstalling databricks-automl-runtime at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-0491bfd1-c162-4440-a6bc-9ab3143b0df2
    Can't uninstall 'databricks-automl-runtime'. No files were found to uninstall.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.3/28.3 MB 35.9 MB/s eta 0:00:00
  Using cached alembic-1.18.3-py3-none-any.whl (262 kB)
  Using cached docker-7.1.0-py3-none-any.whl (147 kB)
  Using cached ml

# Prophet training
- This is an auto-generated notebook.
- To reproduce these results, attach this notebook to Serverless compute, and rerun it.
- Compare trials in the [MLflow experiment](#mlflow/experiments/3257488039771013).
- Clone this notebook into your project folder by selecting **File > Clone** in the notebook toolbar.

In [0]:
import mlflow
import databricks.automl_runtime

target_col = "value"
time_col = "timestamp"
unit = "day"

frequency_quantity = 1


split_col = "split_col"


horizon = 14

## Load Data

In [0]:
import mlflow
import os
import uuid
import shutil
import pandas as pd
import pyspark.pandas as ps

# Create temp directory to download input data from MLflow
input_temp_dir = os.path.join(os.environ["SPARK_LOCAL_DIRS"], "tmp", str(uuid.uuid4())[:8])
os.makedirs(input_temp_dir)

# Download the artifact and read it into a pandas DataFrame
input_data_path = mlflow.artifacts.download_artifacts(run_id="9cdd178b23bf48208756e2d28205e006", artifact_path="data", dst_path=input_temp_dir)

input_file_path = os.path.join(input_data_path, "training_data")
df_loaded = ps.from_pandas(pd.read_parquet(input_file_path))

# Preview data
# display(df_loaded.head(5))

## Aggregate data by `time_col` and `split_col`
Group the data by `time_col` and `split_col`, and take average if there are multiple `target_col` values in the same group.

In [0]:
group_cols = [time_col]
group_cols = group_cols + [split_col]




df_aggregated = df_loaded \
  .groupby(group_cols) \
  .agg(y=(target_col, 'avg')) \
  .reset_index()

# display(df_aggregated.head(5))

## Train Prophet model
- Log relevant metrics to MLflow to track runs
- All the runs are logged under [this MLflow experiment](#mlflow/experiments/3257488039771013)
- Change the model parameters and re-run the training cell to log a different trial to the MLflow experiment

In [0]:
import logging

# disable informational messages from prophet
logging.getLogger("py4j").setLevel(logging.WARNING)

In [0]:
result_columns = ["model_json", "mse", "rmse", "mae", "mape", "mdape", "smape", "coverage"]

def prophet_training(history_pd):
  from hyperopt import hp
  from databricks.automl_runtime.forecast.prophet.forecast import ProphetHyperoptEstimator

  seasonality_mode = ["additive", "multiplicative"]
  search_space =  {
    "changepoint_prior_scale": hp.loguniform("changepoint_prior_scale", -6.9, -0.69),
    "seasonality_prior_scale": hp.loguniform("seasonality_prior_scale", -6.9, 2.3),
    "holidays_prior_scale": hp.loguniform("holidays_prior_scale", -6.9, 2.3),
    "seasonality_mode": hp.choice("seasonality_mode", seasonality_mode)
  }
  country_holidays = None
  run_parallel = True
 
  train_df = history_pd[history_pd[split_col] == "train"]
  split_cutoff = pd.Timestamp(train_df['ds'].max())
  history_pd = history_pd[history_pd[split_col] != "test"]

  hyperopt_estim = ProphetHyperoptEstimator(horizon=horizon, frequency_unit=unit, frequency_quantity=frequency_quantity, metric="smape",interval_width=0.95,
                   country_holidays=country_holidays, search_space=search_space, num_folds=20, max_eval=10, trial_timeout=7200,
                   split_cutoff=split_cutoff,
                   random_state=321419800, is_parallel=run_parallel)

  spark.conf.set("spark.databricks.mlflow.trackHyperopt.enabled", "false")

  results_pd = hyperopt_estim.fit(history_pd)

  spark.conf.set("spark.databricks.mlflow.trackHyperopt.enabled", "true")
 
  return results_pd[result_columns]

In [0]:
import mlflow
from databricks.automl_runtime.forecast.prophet.model import mlflow_prophet_log_model, ProphetModel

with mlflow.start_run(experiment_id="3257488039771013") as mlflow_run:
  mlflow.set_tag("estimator_name", "Prophet")
  mlflow.log_param("interval_width", 0.95)
  mlflow.log_param("random_state", 321419800)
  df_aggregated = df_aggregated.rename(columns={time_col: "ds"})

  forecast_results = prophet_training(df_aggregated.to_pandas())
    
  # Log the metrics to mlflow
  metric_name_map = {"mse": "mean_squared_error", "rmse": "root_mean_squared_error", "mae": "mean_absolute_error",
                     "mape": "mean_absolute_percentage_error", "mdape": "mdape", "smape": "smape", "coverage": "coverage"}
    
  split_prefix = "val_"
  
  
  avg_metrics = forecast_results[metric_name_map.keys()].rename(columns=metric_name_map).mean().to_frame(name="mean_metrics").reset_index()
  avg_metrics["index"] = split_prefix + avg_metrics["index"].astype(str)
  avg_metrics.set_index("index", inplace=True)
  mlflow.log_metrics(avg_metrics.to_dict()["mean_metrics"])
  
  # Create mlflow prophet model
  model_json = forecast_results["model_json"].to_list()[0]
  prophet_model = ProphetModel(model_json, horizon, unit, frequency_quantity, time_col)
  
# Generate sample input dataframe
  sample_input = df_loaded.head(1).to_pandas()
  sample_input[time_col] = pd.to_datetime(sample_input[time_col])
  sample_input.drop(columns=[target_col], inplace=True)
  sample_input.drop(columns=[split_col], inplace=True)

  mlflow_prophet_log_model(prophet_model, sample_input=sample_input)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-0491bfd1-c162-4440-a6bc-9ab3143b0df2/lib/python3.10/site-packages/databricks/automl_runtime/forecast/prophet/forecast.py:149: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["ds"] = pd.to_datetime(df["ds"])
Because the requested parallelism was None or a non-positive value, parallelism will be set to (8), which is Spark's default parallelism (8), or 1, whichever is greater. We recommend setting parallelism explicitly to a positive value because the total of Spark task slots is subject to cluster sizing.
Hyperopt + MLflow integration is feature-flagged off.  To enable automatic tracking in MLflow, set via: `spark.conf.set('spark.databricks.mlflow.trackHyperopt.enabled', 'true')` where `spark` is your Spa

100%|██████████| 10/10 [00:52<00:00,  5.21s/trial, best loss: 1.4128563652170258]


Total Trials: 10: 10 succeeded, 0 failed, 0 cancelled.
12:19:36 - cmdstanpy - INFO - Chain [1] start processing
12:19:36 - cmdstanpy - INFO - Chain [1] done processing
/databricks/python/lib/python3.10/site-packages/prophet/serialize.py:172: FutureWarning: The behavior of Timestamp.utcfromtimestamp is deprecated, in a future version will return a timezone-aware Timestamp with UTC timezone. To keep the old behavior, use Timestamp.utcfromtimestamp(ts).tz_localize(None). To get the future behavior, use Timestamp.fromtimestamp(ts, 'UTC')
  setattr(model, attribute, pd.Timestamp.utcfromtimestamp(model_dict[attribute]).tz_localize(None))
/local_disk0/.ephemeral_nfs/envs/pythonEnv-0491bfd1-c162-4440-a6bc-9ab3143b0df2/lib/python3.10/site-packages/mlflow/pyfunc/__init__.py:3137: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#m

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

In [0]:
avg_metrics
# Uncomment to check the result details. By default, we only show the averaged metric becuase
# displaying the forecast result can take up a lot of storage for large datasets.
# forecast_results.head()

,mean_metrics
index,
val_mean_squared_error,1.167554e+06
val_root_mean_squared_error,1.039208e+03
val_mean_absolute_error,9.249679e+02
val_mean_absolute_percentage_error,1.728533e+00
val_mdape,1.040055e+00
val_smape,1.412856e+00
val_coverage,9.577331e-01
